# **R-CNN (Region-based CNN) Implementation for Object Detection**


---
## **Project Overview**
This project implements the **R-CNN** pipeline, the foundational model for modern object detection.
*   **Selective Search:** Used to propose regions that might contain objects.
*   **Manual IoU:** Intersection over Union logic is written from scratch.
*   **Multi-Task Learning:** A shared VGG-16 backbone with two heads (Classification and Bounding Box Regression).
*   **Post-Processing:** Implementation of Non-Maximum

In [ ]:
!git clone https://github.com/Hulkido/RCNN.git
!unzip -q RCNN/Images.zip -d RCNN/
!unzip -q RCNN/Airplanes_Annotations.zip -d RCNN/

In [8]:
import os 
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import logging
tf.get_logger().setLevel(logging.ERROR)

IMAGE_DIR = "RCNN/Images"
ANNOT_DIR = "RCNN/Airplanes_Annotations"

## **1. Intersection over Union (IoU)**
The IoU is a critical metric for determining the overlap between a predicted box and the ground truth. We implement the formula manually:
$$IoU = \frac{\text{Area of Intersection}}{\text{Area of Union}}$$

In [9]:
# ─── 1. Intersection over Union (IoU) ──────────────────────────────
def get_iou(bb1, bb2):
    """calculating the overlap area of two bounding boxes"""
    x_left = max(bb1['x1'], bb2['x1'])
    y_top = max(bb1['y1'], bb2['y1'])
    x_right = min(bb1['x2'], bb2['x2'])
    y_bottom = min(bb1['y2'], bb2['y2'])

    if x_right < x_left or y_bottom < y_top:
        return 0.0

    intersection_area = (x_right - x_left) * (y_bottom - y_top)
    bb1_area = (bb1['x2'] - bb1['x1']) * (bb1['y2'] - bb1['y1'])
    bb2_area = (bb2['x2'] - bb2['x1']) * (bb2['y2'] - bb2['y1'])

    return intersection_area / float(bb1_area + bb2_area - intersection_area)

## **2. Region Proposal and Dataset Preparation**
In this step, we use **Selective Search** to extract ~2000 proposals per image.
*   **Positive Samples:** Proposals with $IoU > 0.5$ are labeled as Airplanes.
*   **Negative Samples:** Proposals with $IoU < 0.3$ are labeled as Background.
*   **Target Calculation:** We calculate the log-scale offsets ($tx, ty, tw, th$) used for bounding box regression.

## **3. Multi-Task Architecture & Training**
We utilize a pre-trained **VGG-16** backbone. Instead of separate classifiers, we implement a **Multi-Task network**:
1.  **Classification Head:** Categorical Crossentropy loss to distinguish objects from background.
2.  **Regression Head:** Mean Squared Error (MSE) loss to predict the fine-tuned coordinates of the bounding box.

## **4. Inference and Non-Maximum Suppression (NMS)**
During testing, we generate 300 proposals. We predict the class and regression shift for each. We then apply **Non-Maximum Suppression** to eliminate redundant, overlapping bounding boxes, keeping only the most confident detection.

In [ ]:
# ─── 5. TEST & NON-MAXIMUM SUPPRESSION (NMS) ──────────────────────────────────
image = cv2.imread(os.path.join(IMAGE_DIR, "airplane_020.jpg"))
img_disp = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

ss.setBaseImage(image)
ss.switchToSelectiveSearchFast()
ssresults = ss.process()

proposals, boxes = [],[]
for e, result in enumerate(ssresults):
    if e >= 300: break
    x, y, w, h = result
    timage = image[y:y+h, x:x+w]
    proposals.append(cv2.resize(timage, (224, 224), interpolation=cv2.INTER_AREA))
    boxes.append([x, y, w, h])

preds_cls, preds_bbox = model.predict(np.array(proposals), verbose=0)

final_boxes, scores =[], []
for i in range(len(preds_cls)):
    if preds_cls[i][0] > 0.90:
        px, py, pw, ph = boxes[i]
        tx, ty, tw, th = preds_bbox[i]

        gx = (pw * tx) + (px + pw/2.0)
        gy = (ph * ty) + (py + ph/2.0)
        gw = pw * np.exp(tw)
        gh = ph * np.exp(th)

        final_boxes.append([int(gy-gh/2.0), int(gx-gw/2.0), int(gy+gh/2.0), int(gx+gw/2.0)])
        scores.append(preds_cls[i][0])

final_boxes = np.array(final_boxes, dtype=np.float32)
scores = np.array(scores, dtype=np.float32)

if len(final_boxes) > 0:
    selected_indices = tf.image.non_max_suppression(final_boxes, scores, max_output_size=2, iou_threshold=0.3)

    fig, ax = plt.subplots(1, figsize=(8, 8))
    ax.imshow(img_disp)
    for i in selected_indices:
        y1, x1, y2, x2 = final_boxes[i]
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=3, edgecolor='g', facecolor='none')
        ax.add_patch(rect)
        plt.text(x1, y1-5, f"Airplane: {scores[i]:.2f}", color='white', backgroundcolor='green', fontsize=12, fontweight='bold')
    plt.show()

## **5. Final Report Visualizations**
To provide a complete analysis for the report, we generate:
1.  **Selective Search Proposals:** Visualizing the search pattern.
2.  **BBox Regression Visualization:** Comparing the "Sloppy" Selective Search box with the "Shifted" model prediction.
3.  **Training Metrics:** Monitoring loss and accuracy for both heads.

In [ ]:
# ─── VISUALIZATION PIPELINE FOR R-CNN REPORT ──────────────────────────────────
test_img_name = "airplane_020.jpg"
image = cv2.imread(os.path.join(IMAGE_DIR, test_img_name))
img_disp = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
df = pd.read_csv(os.path.join(ANNOT_DIR, test_img_name.replace(".jpg", ".csv")))
coords = df.iloc[0, 0].split(" ")
gt_x1, gt_y1, gt_x2, gt_y2 = int(coords[0]), int(coords[1]), int(coords[2]), int(coords[3])

# Figure 3: Selective Search
fig3, ax3 = plt.subplots(1, figsize=(6, 6))
ax3.imshow(img_disp)
for e, result in enumerate(ssresults):
    if e >= 150: break
    x, y, w, h = result
    ax3.add_patch(patches.Rectangle((x, y), w, h, linewidth=1, edgecolor='yellow', facecolor='none', alpha=0.6))
plt.title("Figure 3: Selective Search Region Proposals"); plt.show()

# Figure 4: Regression
fig4, ax4 = plt.subplots(1, figsize=(6, 6))
ax4.imshow(img_disp)
best_idx = np.argmax(scores)
sx, sy, sw, sh = boxes[best_idx]
fy1, fx1, fy2, fx2 = final_boxes[best_idx]
ax4.add_patch(patches.Rectangle((sx, sy), sw, sh, linewidth=2, edgecolor='red', facecolor='none', label='Sloppy Proposal'))
ax4.add_patch(patches.Rectangle((fx1, fy1), fx2-fx1, fy2-fy1, linewidth=3, edgecolor='green', facecolor='none', label='Shifted Prediction'))
plt.legend(); plt.title("Figure 4: BBox Regression Comparison"); plt.show()

# Figure 6: Training Metrics
epochs_range = range(1, len(history.history['class_output_accuracy']) + 1)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1); plt.plot(epochs_range, history.history['class_output_accuracy'], label='Train'); plt.plot(epochs_range, history.history['val_class_output_accuracy'], label='Val'); plt.title('Classification Accuracy'); plt.legend()
plt.subplot(1, 2, 2); plt.plot(epochs_range, history.history['bbox_output_loss'], label='Train'); plt.plot(epochs_range, history.history['val_bbox_output_loss'], label='Val'); plt.title('BBox Regression Loss (MSE)'); plt.legend()
plt.show()